# 55 Identifying EV, heat pump and solar customers: final models only

A fast version of notebook 50. Notebook 50 compares methods, feature sets and model settings over five rounds (about 2
hours); this notebook fits only the models it settled on and writes the same outputs, in a fraction of the time.
**Run one or the other**, after notebooks 10–45. How the models were chosen, and everything that was tried, is in
notebook 50.

| Technology | Model | Features |
|---|---|---|
| **PV** | Bagging PU, 50 models | Notebook 40's features, peer-relative (B), mid-year change scores (Z) and current-period features (C), plus all signature features (round 2) |
| **EV** | Ensemble: the average percentile of a bagging PU model (50 models) and the night-by-night model | Bagging: notebook 40's features (NumPy version) plus the appliance-rule evidence (A + R, round 4). Night model: `avangrid.ev_nights` (round 5) |
| **HP** | Bagging PU, 20 models | Size-free notebook 40 features, peak load and size-free appliance-rule evidence, plus heat-pump physics features (round 5) |

Every model samples 10 unlabeled customers per labeled one and uses 200-tree gradient boosting, with the same random
seeds as notebook 50. Unlisted customers get out-of-bag scores; listed customers get out-of-fold scores (5 folds).

**Count estimates** (Elkan–Noto, with the known 25% sample share for EV and HP) use the feature sets notebook 50 used
for them: notebook 40's features for EV, HP's round-3 set (B, Z, C, cold-response shape and peak load added), and PV's
model. They set how many unlisted customers go on the EV and HP candidate lists.

**Outputs** (the same as notebook 50's): DuckDB table `technology_predictions`,
`onedrive-export/Data/eda/technology_candidates.csv` and `results/improve_final_scores.parquet`. The last cell
compares the new scores with the table already in the database (from notebook 50 or an earlier run) before replacing
it.

## Setup

`OMP_NUM_THREADS` is set to 1 before scikit-learn is imported: the models are fitted several at a time in separate
processes (`avangrid.pu`), and one thread each is much faster than letting every small model start a thread per core.

In [1]:
import os
os.environ['OMP_NUM_THREADS'] = '1'  # must be set before importing scikit-learn

import time
import warnings

import numpy as np
import polars as pl

import avangrid  # noqa: F401
from avangrid import (appliances, cache, config, customer_features as cfm, db, ev_nights, features, grids, physics,
                      pu, signatures, stability, weather)

warnings.filterwarnings('ignore', category=RuntimeWarning)  # ratios of zero-use hours; those features become NaN
T0 = time.time()


def log(msg):
    print(f'[{time.time() - T0:5.0f}s] {msg}', flush=True)

## Data and features

The cached customers × days × 24 arrays from notebook 40 and the hourly weather, then the feature blocks the final
models use (see notebook 50 for what each block is).

In [2]:
hd = grids.load_hourly(verbose=False)
D, _ = features.load_daily_signals(hd, verbose=False)
n, n_days = hd.shape[0], hd.shape[1]
labels = hd.labels()
not_listed = ~(labels['EV'] | labels['HP'] | labels['PV'])
W = weather.load_weather_hourly()
sig = signatures.load_signatures(hd, D, verbose=False)
log('data loaded')

# Rounds 1-3 blocks (PV's model and the count estimates): notebook 40's features (A0), peer-relative (B),
# mid-year change scores (Z) and each technology's current-period features (C).
pf = features.PeriodFeatures(D, hd.temp)
min_weeks = int(np.ceil(db.sql('SELECT ANY_VALUE(stabilization_days) AS d FROM customer_stability')['d'].item() / 7))
A0, _ = features.customer_features(hd.customer_numbers)
B, _ = pf.to_matrix(pf(0, n_days), {'log_mean_kwh': np.log(hd.customers['mean_kwh_real'].to_numpy())})
cp = stability.tech_change_points(D, hd.kwh, hd.temp, min_weeks, not_listed)
Z = np.column_stack([cp[t]['z'] for t in config.TECHS] + [cp[t]['flag'].astype(float) for t in config.TECHS])


def current_period_block(tech):
    """Current-period features since this technology's flagged change, the before/after differences, period length."""
    flag = cp[tech]['flag']
    change_day = np.where(flag, cp[tech]['week'] * 7, 0)
    cur, bef = pf(change_day, n_days), pf(0, np.maximum(change_day, 1))
    X = np.column_stack([cur[k] for k in cur] + [np.where(flag, cur[k] - bef[k], 0) for k in cur]
                        + [(n_days - change_day) / 7])
    X[~np.isfinite(X)] = np.nan
    return X


def signature_block(*groups):
    return (signatures.to_matrix(sig, groups) if groups else signatures.to_matrix(sig))[0]  # default: the original three


X_pv = np.hstack([A0, B, Z, current_period_block('PV'), signature_block()])
X_hp_count = np.hstack([A0, B, Z, current_period_block('HP'), signature_block('Cold-response shape'),
                        signature_block('Peak load')])

# Rounds 4-5 blocks (EV and HP models): notebook 40's features recomputed in NumPy (A), appliance-rule evidence (R),
# peak load (P) and heat-pump physics (H). "Size-free" drops the features that measure absolute size.
A = cfm.compute(hd.kwh, W['temp'], W['temp_quartile'], hd.dates)
F = appliances.appliance_features(hd.kwh, hd.temp_hourly, hd.dates, hd.weekend)
R_names = [k for k in F if k != 'mean_kwh']
R = np.column_stack([F[k] for k in R_names])
P = signature_block('Peak load')
H = np.column_stack(list(physics.hp_physics_features(hd.kwh, W, hd.dates, F).values()))
SIZE = {'mean_kwh', 'p95_kwh', 'max_kwh', 'summer_midday_kwh', 'summer_night_kwh', 'winter_midday_kwh',
        'heating_slope', 'cooling_slope', 'mild_day_kwh'}
R_SIZE = {'ac_slope', 'ac_added_kw', 'heat_slope', 'heat_kw_at_20f', 'heat_added_kw', 'wh_morning_kwh',
          'pool_day_floor_jump', 'ev_l1_excess_kwh', 'summer_morning_kwh'}
A_sf = A[:, [i for i, nm in enumerate(cfm.NAMES) if nm not in SIZE]]
R_sf = R[:, [i for i, nm in enumerate(R_names) if nm not in R_SIZE]]
X_ev = np.hstack([A, R])
X_hp = np.hstack([A_sf, P, R_sf, H])
night = ev_nights.load_night_features(hd, verbose=False)
log(f'features ready: PV {X_pv.shape[1]}, EV {X_ev.shape[1]}, HP {X_hp.shape[1]} columns')

[    1s] data loaded


[   42s] features ready: PV 88, EV 44, HP 50 columns


## Fit the final models

Each model is fitted once on every listed customer (scores for the unlisted, out-of-bag) and once per fold with a
fifth of the listed customers hidden (out-of-fold scores for the listed).

In [3]:
def final_bagging(X, is_pos, **kw):
    """Scores for every customer: out-of-bag for the unlisted, out-of-fold for the listed. Also returns the
    all-labels model's scores (NaN for the listed)."""
    final, _ = pu.bagging_pu(X, is_pos, seed=0, **kw)
    oof = np.full(n, np.nan)
    for split, lab in pu.hidden_label_splits(is_pos, 'kfold', n_splits=5):
        s, _ = pu.bagging_pu(X, lab, seed=split, **kw)
        oof[is_pos & ~lab] = s[is_pos & ~lab]
    return np.where(is_pos, oof, final), final


def final_night_model(is_pos):
    final = ev_nights.customer_scores(ev_nights.fit_night_models(night, is_pos, seed=0), night)
    oof = np.full(n, np.nan)
    for split, lab in pu.hidden_label_splits(is_pos, 'kfold', n_splits=5):
        s = ev_nights.customer_scores(ev_nights.fit_night_models(night, lab, seed=split), night)
        oof[is_pos & ~lab] = s[is_pos & ~lab]
    return np.where(is_pos, oof, final)


pct = lambda s: pl.Series(s).rank('average').to_numpy() / n  # noqa: E731
pv_score, pv_final = final_bagging(X_pv, labels['PV'], n_models=50)
log('PV done')
ev_bag, _ = final_bagging(X_ev, labels['EV'], n_models=50)
ev_night = final_night_model(labels['EV'])
ev_score = (pct(ev_bag) + pct(ev_night)) / 2
log('EV done')
hp_score, _ = final_bagging(X_hp, labels['HP'])
log('HP done')

[   73s] PV done


[  324s] EV done


[  332s] HP done


## How many customers have each technology

Elkan–Noto's estimates: with c estimated from the data, and (EV and HP) with the known 25% sample share. The candidate
lists take the top unlisted customers up to the known-share estimate minus those listed (EV, HP), or the unlisted
customers scoring at least 0.5 (PV).

In [4]:
counts = []
for tech, X, final in (('EV', A0, None), ('HP', X_hp_count, None), ('PV', X_pv, pv_final)):
    is_pos = labels[tech]
    p, c_est, g = pu.elkan_noto(X, is_pos, seed=0)
    counts.append({'technology': tech, 'listed_in_analysis': int(is_pos.sum()), 'estimated_c': float(c_est),
                   'est_total_estimated_c': float(p.sum()),
                   'est_total_known_c_0.25': float(np.clip(g / 0.25, 0, 1).sum()) if tech != 'PV' else None,
                   'unlisted_score_ge_0.5': int(((final >= 0.5) & ~is_pos).sum()) if final is not None else None})
counts = pl.DataFrame(counts)
c = {r['technology']: r for r in counts.iter_rows(named=True)}
size = {'EV': round(c['EV']['est_total_known_c_0.25'] - c['EV']['listed_in_analysis']),
        'HP': round(c['HP']['est_total_known_c_0.25'] - c['HP']['listed_in_analysis']),
        'PV': c['PV']['unlisted_score_ge_0.5']}
with pl.Config(tbl_hide_dataframe_shape=True, float_precision=3, tbl_width_chars=160):
    print(counts)
print(f'Candidate list sizes: {size}')

┌────────────┬────────────────────┬─────────────┬───────────────────────┬────────────────────────┬───────────────────────┐
│ technology ┆ listed_in_analysis ┆ estimated_c ┆ est_total_estimated_c ┆ est_total_known_c_0.25 ┆ unlisted_score_ge_0.5 │
│ ---        ┆ ---                ┆ ---         ┆ ---                   ┆ ---                    ┆ ---                   │
│ str        ┆ i64                ┆ f64         ┆ f64                   ┆ f64                    ┆ i64                   │
╞════════════╪════════════════════╪═════════════╪═══════════════════════╪════════════════════════╪═══════════════════════╡
│ EV         ┆ 179                ┆ 0.027       ┆ 5783.563              ┆ 714.906                ┆ null                  │
│ HP         ┆ 86                 ┆ 0.008       ┆ 8424.153              ┆ 346.497                ┆ null                  │
│ PV         ┆ 389                ┆ 0.773       ┆ 434.664               ┆ null                   ┆ 61                    │
└────────────┴──

## Save, and compare with the previous results

`technology_predictions` has one row per customer: each technology's score, percentile among all customers, whether
listed, and rank among unlisted customers (plus the EV ensemble's two parts). Before replacing the table, its scores
are compared with the existing one; after a run of notebook 50 on the same data they should match exactly.

In [5]:
df = hd.customers.select('CUSTOMER_NUMBER').with_columns(
    pl.Series('ev_score', ev_score), pl.Series('ev_bagging_score', ev_bag), pl.Series('ev_night_score', ev_night),
    pl.Series('hp_score', hp_score), pl.Series('pv_score', pv_score),
    *[pl.Series(f'{t.lower()}_listed', labels[t]) for t in config.TECHS])
for t in ['ev', 'hp', 'pv']:
    df = df.with_columns((pl.col(f'{t}_score').rank('average') / n).alias(f'{t}_percentile'))
    df = df.with_columns(pl.when(pl.col(f'{t}_listed')).then(None)
                         .otherwise(pl.col(f'{t}_score').rank('ordinal', descending=True).over(f'{t}_listed'))
                         .alias(f'{t}_rank_among_unlisted'))
df = df.select('CUSTOMER_NUMBER',
               'ev_score', 'ev_percentile', 'ev_listed', 'ev_rank_among_unlisted', 'ev_bagging_score', 'ev_night_score',
               'hp_score', 'hp_percentile', 'hp_listed', 'hp_rank_among_unlisted',
               'pv_score', 'pv_percentile', 'pv_listed', 'pv_rank_among_unlisted').sort('CUSTOMER_NUMBER')
cands = pl.concat([df.filter(~pl.col(f'{t.lower()}_listed')).sort(f'{t.lower()}_score', descending=True).head(k)
                   .select('CUSTOMER_NUMBER', pl.lit(t).alias('technology'), pl.col(f'{t.lower()}_score').alias('score'),
                           pl.col(f'{t.lower()}_rank_among_unlisted').alias('rank'))
                   for t, k in size.items()])

if db.table_exists('technology_predictions'):
    old = db.sql('SELECT * FROM technology_predictions')
    j = old.join(df, on='CUSTOMER_NUMBER', how='full', suffix='_new')
    print(f'Customers: {old.height:,} before, {df.height:,} now, {j.filter(pl.col("CUSTOMER_NUMBER").is_null() | pl.col("CUSTOMER_NUMBER_new").is_null()).height} not in both')
    for col in ('ev_score', 'ev_bagging_score', 'ev_night_score', 'hp_score', 'pv_score'):
        d = (j[col] - j[f'{col}_new']).abs()
        print(f'  {col:17} largest difference from the previous table: {d.max():.2e}')
    old_c = config.EDA_DIR / 'technology_candidates.csv'
    if old_c.exists():
        prev = pl.read_csv(old_c)
        for t in config.TECHS:
            a = set(prev.filter(pl.col('technology') == t)['CUSTOMER_NUMBER'])
            b = set(cands.filter(pl.col('technology') == t)['CUSTOMER_NUMBER'])
            print(f'  {t} candidates: {len(b)} now, {len(a)} before, {len(a & b)} in both')

with db.connect(read_only=False) as con:
    con.register('df', df)
    con.execute('CREATE OR REPLACE TABLE technology_predictions AS SELECT * FROM df')
cands.write_csv(config.EDA_DIR / 'technology_candidates.csv')
cache.save_results('improve_final_scores', df.select(
    'CUSTOMER_NUMBER', pl.col('ev_score').alias('ev_score_v2'), 'ev_bagging_score', 'ev_night_score',
    pl.col('hp_score').alias('hp_score_v2'), 'ev_listed', 'hp_listed'))
log(f'technology_predictions refreshed ({df.height:,} customers); candidates: {size}')

Customers: 33,310 before, 33,310 now, 0 not in both
  ev_score          largest difference from the previous table: 0.00e+00
  ev_bagging_score  largest difference from the previous table: 0.00e+00
  ev_night_score    largest difference from the previous table: 0.00e+00
  hp_score          largest difference from the previous table: 0.00e+00
  pv_score          largest difference from the previous table: 0.00e+00
  EV candidates: 536 now, 536 before, 536 in both
  HP candidates: 260 now, 260 before, 260 in both
  PV candidates: 61 now, 61 before, 61 in both
[  336s] technology_predictions refreshed (33,310 customers); candidates: {'EV': 536, 'HP': 260, 'PV': 61}
